In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="your-api-key")
project = rf.workspace("amsroboflow").project("football_detection-ybjkh")
version = project.version(11)
dataset = version.download("yolo26")


In [ ]:
!pip install -q -U ultralytics

In [ ]:
# ============================================================
# FOOTBALLVISION — YOLO26 SMALL TRAINING
# Dataset is already in YOLO format. No annotation conversion.
# ============================================================

!pip install -q -U ultralytics pyyaml

from pathlib import Path
import yaml
import torch
import ultralytics
from ultralytics import YOLO

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

INPUT_ROOT = Path("/kaggle/working")
WORK_DIR = Path("/kaggle/working/FootballVision")

EXPECTED_CLASSES = [
    "ball",
    "goalkeeper",
    "player",
    "referee",
]

# Default: start from official pretrained YOLO26 Small weights.
START_WEIGHTS = "yolo26s.pt"

# OPTIONAL:
# If you uploaded your previous best.pt as a Kaggle dataset,
# replace START_WEIGHTS with its full path, for example:
# START_WEIGHTS = "/kaggle/input/your-checkpoint/best.pt"
#
# This initializes from those trained weights, but does not
# restore the previous run's optimizer state or epoch number.

# Training configuration
EPOCHS = 50
IMAGE_SIZE = 960
BATCH_SIZE = 4
PATIENCE = 12

RUNS_DIR = WORK_DIR / "runs"
DATA_YAML = Path("/kaggle/working/footballvision_data.yaml")

# ------------------------------------------------------------
# 2. CHECK GPU
# ------------------------------------------------------------

print("=" * 60)
print("ENVIRONMENT")
print("=" * 60)

print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Enable a GPU in Kaggle Notebook Settings "
        "and rerun this cell."
    )

print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 3. FIND THE EXISTING YOLO DATASET
# ------------------------------------------------------------

def normalized_names(config):
    names = config.get("names", {})

    if isinstance(names, dict):
        names = [
            names[key]
            for key in sorted(names, key=lambda value: int(value))
        ]

    return [str(name).strip().lower() for name in names]


def valid_dataset_root(root):
    required_splits = ["train", "valid", "test"]

    return all(
        (root / split / "images").is_dir()
        and (root / split / "labels").is_dir()
        for split in required_splits
    )


yaml_candidates = sorted(
    set(INPUT_ROOT.rglob("data.yaml"))
    | set(INPUT_ROOT.rglob("data.yml"))
)

matches = []

for yaml_path in yaml_candidates:
    try:
        source_config = yaml.safe_load(yaml_path.read_text())
    except Exception:
        continue

    if normalized_names(source_config or {}) != EXPECTED_CLASSES:
        continue

    # Search upward from data.yaml until we find the dataset root.
    root = yaml_path.parent
    depth = 0

    while root == INPUT_ROOT or INPUT_ROOT in root.parents:
        if valid_dataset_root(root):
            matches.append((depth, root, yaml_path))
            break

        if root == INPUT_ROOT:
            break

        root = root.parent
        depth += 1

if not matches:
    raise FileNotFoundError(
        "Could not find a YOLO dataset with train/valid/test "
        "images and labels, and the expected four classes.\n"
        "Check that your dataset is attached to this Kaggle notebook."
    )

# Prefer the closest matching data.yaml and dataset root.
matches.sort(
    key=lambda item: (
        item[0],
        0 if "football" in str(item[1]).lower() else 1
    )
)

_, DATASET_ROOT, SOURCE_YAML = matches[0]

print("\nDataset found:")
print("Root:", DATASET_ROOT)
print("Original YAML:", SOURCE_YAML)

# ------------------------------------------------------------
# 4. WRITE A KAGGLE-CORRECTED YAML
# ------------------------------------------------------------

# Kaggle input files are read-only, so write the corrected
# configuration to /kaggle/working instead.

dataset_config = {
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 4,
    "names": {
        0: "ball",
        1: "goalkeeper",
        2: "player",
        3: "referee",
    },
}

DATA_YAML.write_text(
    yaml.safe_dump(dataset_config, sort_keys=False)
)

print("\nCorrected dataset YAML:")
print(DATA_YAML.read_text())

# ------------------------------------------------------------
# 5. VERIFY IMAGE AND LABEL COUNTS
# ------------------------------------------------------------

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

print("=" * 60)
print("DATASET CHECK")
print("=" * 60)

for split in ["train", "valid", "test"]:
    image_dir = DATASET_ROOT / split / "images"
    label_dir = DATASET_ROOT / split / "labels"

    images = [
        path for path in image_dir.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    ]

    labels = list(label_dir.rglob("*.txt"))

    print(
        f"{split.upper()}: "
        f"{len(images)} images, {len(labels)} label files"
    )

    if not images or not labels:
        raise ValueError(
            f"The {split} split is empty or its labels are missing."
        )

    if len(images) != len(labels):
        raise ValueError(
            f"Image/label count mismatch in {split}. "
            "Inspect the dataset before training."
        )

print("\nDataset checks passed.")

# ------------------------------------------------------------
# 6. LOAD MODEL
# ------------------------------------------------------------

print("\nLoading model:", START_WEIGHTS)

model = YOLO(START_WEIGHTS)

print("Model loaded.")
print("Beginning YOLO26 Small fine-tuning...")

# ------------------------------------------------------------
# 7. TRAIN
# ------------------------------------------------------------

results = model.train(
    data=str(DATA_YAML),

    epochs=EPOCHS,
    patience=PATIENCE,

    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,

    device=0,
    workers=2,

    project=str(RUNS_DIR),
    name="yolo26s_football_kaggle",
    exist_ok=False,

    save=True,
    save_period=5,
    plots=True,
    seed=42,
)

# ------------------------------------------------------------
# 8. REPORT CHECKPOINT LOCATIONS
# ------------------------------------------------------------

save_dir = Path(results.save_dir)

print("\n" + "=" * 60)
print("TRAINING FINISHED")
print("=" * 60)

print("Results directory:", save_dir)
print("Best checkpoint:", save_dir / "weights" / "best.pt")
print("Last checkpoint:", save_dir / "weights" / "last.pt")
print("Training metrics:", save_dir / "results.csv")

In [ ]:
from pathlib import Path
from ultralytics import YOLO

WEIGHTS = Path(
    "/kaggle/input/datasets/anishnagargoje/footballviddeo/best (1).pt"
)

DATA_YAML = Path("/kaggle/working/football_detection-11/data.yaml")

assert WEIGHTS.exists(), f"Checkpoint not found: {WEIGHTS}"
assert DATA_YAML.exists(), f"Dataset YAML not found: {DATA_YAML}"

# Load the trained model. This does not retrain it.
model = YOLO(str(WEIGHTS))

# Evaluate on the held-out test split.
metrics = model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=960,
    batch=4,
    device=0,
    plots=True,
    project="/kaggle/working/FootballVision/runs",
    name="yolo26s_test_evaluation",
    exist_ok=True,
)

print("\nTEST RESULTS")
print("Precision:", round(float(metrics.box.mp), 4))
print("Recall:", round(float(metrics.box.mr), 4))
print("mAP@50:", round(float(metrics.box.map50), 4))
print("mAP@50-95:", round(float(metrics.box.map), 4))

print("\nPER-CLASS mAP@50-95")
for class_id, score in enumerate(metrics.box.maps):
    print(f"{model.names[class_id]}: {score:.4f}")